In [1]:
#Build data/player_bios.csv for the players in data/clean.csv.
# Run on YOUR laptop (stats.nba.com usually blocks cloud servers). Needs: pip install nba_api pandas
import time
import pandas as pd
from nba_api.stats.endpoints import commonteamroster

In [2]:
SEASON = "2025"          # WNBA seasons are the plain year; if you get an error, check help(commonteamroster.CommonTeamRoster)
shots = pd.read_csv("wnba_clean.csv")
team_ids = sorted(shots["team_id"].unique())
my_players = set(shots["player_id"].unique())
 
frames = []
for tid in team_ids:
    r = commonteamroster.CommonTeamRoster(team_id=int(tid), season=SEASON,
                                          league_id_nullable="10", timeout=60)
    frames.append(r.get_data_frames()[0])      # first table = the roster (the second is coaches)
    time.sleep(1)                              # be polite to the server

In [3]:
raw = pd.concat(frames, ignore_index=True)
raw.to_csv("wnba_raw.csv", index=False)   # keep exactly what the API returned
print(raw.columns.tolist())

['TeamID', 'SEASON', 'LeagueID', 'PLAYER', 'NICKNAME', 'PLAYER_SLUG', 'NUM', 'POSITION', 'HEIGHT', 'WEIGHT', 'BIRTH_DATE', 'AGE', 'EXP', 'SCHOOL', 'PLAYER_ID', 'HOW_ACQUIRED']


In [4]:
def height_to_inches(h):
    # "6-2" -> 74
    try:
        ft, inch = str(h).split("-")
        return int(ft) * 12 + int(inch)
    except Exception:
        return None
 

In [5]:
bios = pd.DataFrame({
    "player_id": raw["PLAYER_ID"].astype(int),
    "height_in": raw["HEIGHT"].map(height_to_inches),
    "weight_lb": pd.to_numeric(raw["WEIGHT"], errors="coerce"),
    "position":  raw["POSITION"],
})
bios = bios[bios["player_id"].isin(my_players)]
bios = bios.drop_duplicates("player_id")       # traded players show up on more than one roster
bios.to_csv("player_bios.csv", index=False)

In [6]:
print("players in shots:", len(my_players))
print("players with a bio:", len(bios))
print("missing player_ids:", sorted(my_players - set(bios["player_id"])))
print(bios[["height_in", "weight_lb"]].describe().round(1))
print(bios["position"].value_counts(dropna=False))

players in shots: 179
players with a bio: 160
missing player_ids: [np.int64(204325), np.int64(1627669), np.int64(1628297), np.int64(1628304), np.int64(1628317), np.int64(1628920), np.int64(1629479), np.int64(1630115), np.int64(1641641), np.int64(1642240), np.int64(1642295), np.int64(1642301), np.int64(1642763), np.int64(1642768), np.int64(1642773), np.int64(1642774), np.int64(1642799), np.int64(1642813), np.int64(1642820)]
       height_in  weight_lb
count      160.0      127.0
mean        72.6      172.9
std          3.2       22.5
min         66.0      130.0
25%         70.0      158.5
50%         73.0      175.0
75%         75.0      188.0
max         81.0      245.0
position
G      72
F      50
C      16
G-F     9
F-C     6
C-F     4
F-G     3
Name: count, dtype: int64


In [7]:

shots = pd.read_csv("wnba_clean.csv")
bios = pd.read_csv("player_bios.csv")

assert bios["player_id"].is_unique, "bios must have one row per player"

m = shots.merge(bios, on="player_id", how="left", validate="m:1", indicator=True)

unmatched = m.loc[m["_merge"] == "left_only"]
print("players in shots:", shots["player_id"].nunique())
print("players with bios:", m.loc[m["_merge"] == "both", "player_id"].nunique())
print("unmatched players:", unmatched["player_id"].nunique())
print("shots lost to no bio:", len(unmatched), "of", len(m))
print(m[["height_in", "weight_lb", "position"]].isna().sum())

m = m.drop(columns="_merge")
m.to_csv("shots_with_bios.csv", index=False)   # a NEW file; clean.csv stays untouched
pd.read_csv("shots_with_bios.csv").head(10)

players in shots: 179
players with bios: 160
unmatched players: 19
shots lost to no bio: 332 of 38535
height_in     332
weight_lb    6566
position      332
dtype: int64


,game_id,game_event_id,player_id,player_name,team_id,team_name,period,action_type,shot_type,shot_zone_basic,...,made_flag,game_date,htm,vtm,seconds_left,is_overtime,is_backcourt,height_in,weight_lb,position
0,1022500001,7,1631009,Rhyne Howard,1611661330,Atlanta Dream,1,Jump Shot,3PT Field Goal,Above the Break 3,...,1,2025-05-16,WAS,ATL,583,0,0,74.0,175.0,G
1,1022500001,9,1628279,Brittney Sykes,1611661322,Washington Mystics,1,Pullup Jump shot,2PT Field Goal,Mid-Range,...,0,2025-05-16,WAS,ATL,569,0,0,69.0,154.0,G
2,1022500001,11,1628280,Brionna Jones,1611661330,Atlanta Dream,1,Jump Shot,3PT Field Goal,Above the Break 3,...,0,2025-05-16,WAS,ATL,552,0,0,75.0,215.0,F
3,1022500001,13,1642785,Sonia Citron,1611661322,Washington Mystics,1,Running Jump Shot,2PT Field Goal,Mid-Range,...,0,2025-05-16,WAS,ATL,544,0,0,73.0,NaN,G
4,1022500001,26,1642792,Kiki Iriafen,1611661322,Washington Mystics,1,Layup Shot,2PT Field Goal,In The Paint (Non-RA),...,0,2025-05-16,WAS,ATL,485,0,0,75.0,NaN,F
5,1022500001,30,1628279,Brittney Sykes,1611661322,Washington Mystics,1,Pullup Jump shot,2PT Field Goal,Mid-Range,...,0,2025-05-16,WAS,ATL,481,0,0,69.0,154.0,G
6,1022500001,42,1631009,Rhyne Howard,1611661330,Atlanta Dream,1,Jump Shot,3PT Field Goal,Above the Break 3,...,0,2025-05-16,WAS,ATL,420,0,0,74.0,175.0,G
7,1022500001,45,1642792,Kiki Iriafen,1611661322,Washington Mystics,1,Cutting Layup Shot,2PT Field Goal,Restricted Area,...,1,2025-05-16,WAS,ATL,405,0,0,75.0,NaN,F
8,1022500001,57,1642792,Kiki Iriafen,1611661322,Washington Mystics,1,Layup Shot,2PT Field Goal,In The Paint (Non-RA),...,1,2025-05-16,WAS,ATL,385,0,0,75.0,NaN,F
9,1022500001,59,1628269,Nia Coffey,1611661330,Atlanta Dream,1,Jump Shot,3PT Field Goal,Left Corner 3,...,1,2025-05-16,WAS,ATL,368,0,0,73.0,182.0,F
